In [1]:
from typing import Annotated, Sequence, TypedDict
from langchain.chat_models import init_chat_model
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_core.tools import tool
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import InMemorySaver
from pydantic import BaseModel
from dotenv import load_dotenv 

load_dotenv()  

True

In [2]:
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]
    next_node : str 

In [3]:
llm = init_chat_model(model_provider="google_genai", 
                        model ="gemini-2.5-flash",
                        temperature=0.7,
                        max_output_tokens=1024,
                        top_p=0.95,
                        top_k=40,
                        timeout = 30,
                        max_retries = 3,
                        load_env=True)

Unexpected argument 'load_env' provided to ChatGoogleGenerativeAI.
/home/lakshay/LangGraph/.venv/lib/python3.12/site-packages/langchain/chat_models/base.py:544: UserWarning: WARNING! load_env is not default parameter.
                load_env was transferred to model_kwargs.
                Please confirm that load_env is what you intended.
  return creator_func(model=model, **kwargs)


In [2]:
checkpointer = InMemorySaver()

In [4]:
@tool
def add(a: float , b:float) -> float:
    """Add two numbers"""
    return a + b

In [5]:
math_agent = create_react_agent(
    llm, 
    tools=[add], 
    prompt="You are a precise calculation assistant. Use your tools for math."
)

/tmp/ipykernel_16502/3476448330.py:1: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  math_agent = create_react_agent(


In [6]:
writer_agent = create_react_agent(
    llm,
    tools = [],
    prompt="You are a technical educator. Explain numerical results in plain terms."
)

/tmp/ipykernel_16502/3026759365.py:1: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  writer_agent = create_react_agent(


In [7]:
class RouterDecision(BaseModel):
    next : str

supervisor_llm = llm.with_structured_output(RouterDecision)

def supervisor_node(state: AgentState) -> dict:

    prompt = (
        "Given the conversation below, decide who should act next: "
        "'math' (to calculate), 'writer' (to format/explain), or 'FINISH'.\n\n"
        f"Messages: {state['messages']}"
    )

    decision = supervisor_llm.invoke(prompt)
    return {"next_node" : decision.next}

In [8]:
def math_node(state: AgentState) -> dict:
    result = math_agent.invoke({"messages":state['messages']})
    return {"messages" : [result["message"][-1]]}

In [9]:
def writer_node(state: AgentState) -> dict:
    result = writer_agent.invoke({"messages": state["messages"]})
    return {"messages": [result["messages"][-1]]}

In [10]:
builder = StateGraph(AgentState)

In [11]:
builder.add_node("supervisor", supervisor_node)
builder.add_node("math", math_node)
builder.add_node("writer", writer_node)

In [12]:
builder.add_edge(START, "supervisor")

In [13]:
builder.add_conditional_edges(
    "supervisor",
    lambda state: state["next_node"],
    {
        "math": "math",
        "writer": "writer",
        "FINISH": END,
    },
)

In [ ]:
builder.add_edge("math", "supervisor")
builder.add_edge("writer", "supervisor")

multi_agent_graph = builder.compile(checkpointer = checkpointer)

In [ ]:
config = {"configurable": {"thread_id": "session-1"}}

In [ ]:
inputs = {
    "messages": [HumanMessage(content="Calculate 948 + 352, then explain the significance of 1300 in a brief sentence.")]
}
for chunk in multi_agent_graph.stream(stream_mode="updates", input=inputs, config=config):
    print(chunk)

In [16]:
print(multi_agent_graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	supervisor(supervisor)
	math(math)
	writer(writer)
	__end__([<p>__end__</p>]):::last
	__start__ --> supervisor;
	math --> supervisor;
	supervisor -. &nbsp;FINISH&nbsp; .-> __end__;
	supervisor -.-> math;
	supervisor -.-> writer;
	writer --> supervisor;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

